# spaCy Tokenization Deep Dive
Tokenization is the process of splitting text into meaningful units (tokens). spaCy's tokenizer works in two stages:
1. **Whitespace split** — splits on spaces first
2. **Prefix/suffix/infix rules** — further splits punctuation, currency symbols, etc.

This produces linguistically meaningful tokens rather than naive character-based splits.

### `spacy.blank("en")` vs `spacy.load("en_core_web_sm")`
| | `spacy.blank()` | `spacy.load()` |
|---|---|---|
| Pipeline components | None (tokenizer only) | tok2vec, tagger, parser, NER, lemmatizer |
| Use case | Fast tokenization, custom pipelines | Full NLP analysis |
| Model size | Negligible | ~12 MB (sm), ~43 MB (md) |

Notice that `$` is separated from `2` in the output — spaCy treats currency symbols as standalone tokens.

In [1]:
import spacy

In [2]:
nlp = spacy.blank("en")

doc = nlp("Dr. Strange loves pav bhaji of mumbai as it costs only 2$ per plate.")
for token in doc:
    print(token.text)

### `spacy.blank("en")` — Tokenizer Only
Creates a blank English pipeline with **only the tokenizer** — no POS tagger, no NER, no parser. Useful when you need fast tokenization without the overhead of a full model.

### `spacy.load("en_core_web_sm")` — Full Pre-trained Pipeline
Loads a small English model with all pipeline components active: **tok2vec → tagger → parser → attribute_ruler → lemmatizer → NER**. Use this when you need POS tags, lemmas, or named entities.

###  The `Doc` Object & Token Indexing
A `Doc` is a sequence of `Token` objects. You can access individual tokens by integer index (`doc[0]`, `doc[1]`). Each `Token` carries rich linguistic metadata — inspect all available attributes with `dir(token)`.

In [3]:
doc[0]

In [4]:
token = doc[1]
token.text

`dir(token)` reveals all attributes and methods on a `Token`. Key ones to remember: `.text`, `.lemma_`, `.pos_`, `.is_alpha`, `.is_stop`, `.like_email`, `.like_url`, `.is_currency`.

In [5]:
dir(token)

Checking types confirms: `nlp` is a `spacy.lang.en.English` pipeline object, and `doc` is a `spacy.tokens.doc.Doc`. Understanding these types is essential when extending spaCy with custom components.

In [6]:
type(nlp)

In [7]:
type(doc)

###  Span Objects — Slicing a Doc
A `Span` is a slice of a `Doc` (a contiguous sequence of tokens). Use Python list-slice notation `doc[start:end]`. Spans are useful for extracting named entities, noun chunks, or sentence fragments.

In [8]:
span = doc[0:5]
span

In [9]:
type(span)

###  Token Attributes — Boolean Flags
spaCy tokens carry boolean flags that allow rule-based filtering. Key flags:
- `is_alpha` — True if all characters are alphabetic
- `is_digit` — True if all characters are digits
- `like_num` — True for number-like tokens (words like `"two"` also return True)
- `is_currency` — True for currency symbols (`$`, `€`, etc.)
- `is_punct` — True for punctuation

Note: `"two".is_digit` is **False** (not digits), but `"two".like_num` is **True** (semantically numeric).

In [10]:
doc = nlp("Tony gave two $ to Peter.")

In [11]:
token0 = doc[0]
token0

In [12]:
token0.is_alpha

In [13]:
token2 = doc[2]
token2

In [14]:
token2.is_digit

In [15]:
token2.is_alpha

In [16]:
token2.like_num

In [17]:
token3 = doc[3]
token3

In [18]:
type(token3)

In [19]:
token2.is_alpha

In [20]:
token3.is_currency

Printing all token attributes in a formatted table helps you see the interplay of flags. Notice: `$` is `is_currency=True` and `is_alpha=False`; `two` is `like_num=True` even though `is_digit=False`.

In [21]:
for token in doc:
    print(token, "==>", "index: ", token.i, "is_alpha:", token.is_alpha, 
          "is_punct:", token.is_punct, 
          "like_num:", token.like_num,
          "is_currency:", token.is_currency,
         )

###  Real-World Use Case: Extracting Emails from a Student File
Using `token.like_email`, we can scan any unstructured text and extract email addresses — no regex needed. spaCy's tokenizer intelligently keeps `virat@kohli.com` as a single token.

In [22]:
with open("students.txt") as f:
    text = f.readlines()
text

Lines are joined into a single string with spaces — spaCy processes a single `Doc`, so all text must be one string. The `\n` characters inside are handled gracefully by the tokenizer.

In [23]:
text = " ".join(text)
text

`token.like_email` returns `True` for any token matching the email pattern (`word@domain.tld`). List comprehension collects only those tokens — a clean, readable approach for information extraction.

In [24]:
doc = nlp(text)
emails = []

for tokn in doc:
    if tokn.like_email:
        emails.append(tokn.text)

emails

###  Customizing the Tokenizer — Special Cases
`nlp.tokenizer.add_special_case()` lets you override how a specific word is tokenized. Here, `"gimme"` → `["gim", "me"]`. The ORTH rule ensures the original text is preserved when rebuilding the string.

In [25]:
from spacy.symbols import ORTH
nlp = spacy.blank("en")

doc = nlp("gimme double cheese extra large healthy pizza")
tokens = [token.text for token in doc]
tokens

After adding the special case, `"gimme"` is split into `["gim", "me"]`. This is useful for domain-specific contractions (e.g., `"gonna"`, `"wanna"`, `"lemme"`) that standard rules don't handle.

In [26]:
nlp.tokenizer.add_special_case("gimme", [
    {ORTH: "gim"},
    {ORTH: "me"}             # even if we are using special case that should not entirely change the original text
])

doc = nlp("gimme double cheese extra large healthy pizza")
tokens = [token.text for token in doc]
tokens

###  Sentence Tokenization (Segmentation) — Requires a Pipeline Component
`doc.sents` requires sentence boundaries to be set. With `spacy.blank()`, no parser exists to detect boundaries — you must add the `sentencizer` component explicitly.

 **Expected Error:** The blank pipeline has no component that sets sentence boundaries (`is_sent_start`). Running `doc.sents` throws a `ValueError`. Fix: add `'sentencizer'` to the pipeline.

In [27]:
doc = nlp("Dr. Strange loves pav bhaji of mumbai. Hulk loves chat of delhi")
for sentence in doc.sents:
    print(sentence)

`nlp.pipeline` returns `[]` — confirming the blank model has no components loaded. This is why `doc.sents` fails above.

In [28]:
nlp.pipeline

`nlp.add_pipe('sentencizer')` adds a rule-based sentence boundary detector. The `Sentencizer` splits on sentence-ending punctuation (`.`, `!`, `?`). Once added, `doc.sents` works correctly.

In [29]:
nlp.add_pipe('sentencizer')

After adding the sentencizer, `doc.sents` correctly yields 2 sentence spans. The text must be re-processed through `nlp()` to apply the new component.

In [30]:
doc = nlp("Dr. Strange loves pav bhaji of mumbai. Hulk loves chat of delhi")
for sentence in doc.sents:
    print(sentence)